# Target allocation, step by step with Trajan

Run this notebook after loading `st` (a SynapseTable) and `cell_types` (a Polars DataFrame with root_id, class, subclass, cluster). The supplied cell table must represent the eligible target population, including cells with no observed synapses. IDs must be unique and non-null.

Each intermediate result is a Trajan ConnectivityTable. Population counts and the final wide export use Polars; no helper script is required. Class fractions span exc/inh. Subclass and cluster fractions are conditional on target class. Cluster keys include their parent subclass so reused cluster names stay distinct. Missing child labels get an explicit bucket.


In [ ]:
import polars as pl

# Change this to the ID column in your cell type table.
cell_id_col = "root_id"
cells = cell_types.select(cell_id_col, "class", "subclass", "cluster")
assert cells[cell_id_col].null_count() == 0
assert cells[cell_id_col].n_unique() == cells.height

# Skip registration if these columns are already attached to st.
if "class_post" not in st.build_lazy().collect_schema().names():
    st.add_cell_annotation("taxonomy", cells, cell_id_col=cell_id_col)

# Declare grouping labels on the synapse table. Add these once.
if "target_subclass" not in st.build_lazy().collect_schema().names():
    st.add_expression("target_subclass", pl.col("subclass_post").cast(pl.String).fill_null("__unlabeled__"))
if "target_cluster" not in st.build_lazy().collect_schema().names():
    st.add_expression("target_cluster", pl.concat_str(
        pl.col("class_post").cast(pl.String),
        pl.col("subclass_post").cast(pl.String).fill_null("__unlabeled__"),
        pl.col("cluster_post").cast(pl.String).fill_null("__unlabeled__"),
        separator=" / ",
    ))


## Select the synapses to analyze

Start with all synapses. For a compartment or distance analysis, change `analysis` and rerun the downstream cells. `dist_to_root` is the supplied geodesic distance; use an ordinary synapse filter. Population counts below stay fixed. Autapses are retained and population counts include self if eligible.

The baseline counts below expose unresolved endpoints and missing target classes. The allocation tables use only complete endpoints and known exc/inh targets. Neurons with no classified targets are restored in the final export with undefined fractions.


In [ ]:
analysis = st
# analysis = st.filter(pl.col("dist_to_root") < cutoff)
# analysis = st.filter(pl.col("tag") == "spine")

pre = st.pre_col
baseline = analysis.build_lazy().filter(pl.col(pre).is_not_null()).group_by(pre).agg(
    pl.len().alias("n_syn_total"),
    pl.col(st.post_col).is_null().sum().alias("n_syn_missing_post"),
    (~pl.col("class_post").is_in(["exc", "inh"]).fill_null(False)).sum().alias("n_syn_unclassified"),
).collect().sort(pre)

source = analysis.filter(
    pl.col(pre).is_not_null()
    & pl.col(st.post_col).is_not_null()
    & pl.col("class_post").is_in(["exc", "inh"])
)

# Applied to individual synapses, so mean_size is a synapse-level mean.
agg = {
    "size_sum": pl.sum("size"),
    "n_size": pl.col("size").count(),
    "mean_size": pl.mean("size"),
}


## Class allocation and tag fractions

`type_edgelist` permits a cell ID on the pre axis and a type label on the post axis. It supplies n_syn automatically. `fraction_by="tag"` produces a struct of count fractions within each neuron × target-class group; its null bucket stays explicit. Size allocation uses total size, not mean size. Null sizes are skipped in means and size sums; n_size records the number measured.


In [ ]:
class_ct = source.type_edgelist(
    pre_col=pre,
    post_col="class_post",
    agg=agg,
    fraction_by="tag",
)
class_count_fraction = class_ct.normalize(by="pre", values="n_syn")
class_size_fraction = class_ct.normalize(by="pre", values="size_sum")
class_ct.df


In [ ]:
exc_tag_fraction = (
    class_ct.filter(pl.col("class_post") == "exc")
    .df.select(pre, "tag")
    .unnest("tag")
)
exc_tag_fraction


## Subclass and cluster allocation

Filter to exc/inh before aggregating and normalizing so fractions are conditional on class. The same fraction_by argument provides compartment fractions within every target subclass/cluster.


In [ ]:
exc_source = source.filter(pl.col("class_post") == "exc")
inh_source = source.filter(pl.col("class_post") == "inh")

exc_subclass_ct = exc_source.type_edgelist(pre, "target_subclass", agg=agg, fraction_by="tag")
inh_subclass_ct = inh_source.type_edgelist(pre, "target_subclass", agg=agg, fraction_by="tag")
exc_cluster_ct = exc_source.type_edgelist(pre, "target_cluster", agg=agg, fraction_by="tag")
inh_cluster_ct = inh_source.type_edgelist(pre, "target_cluster", agg=agg, fraction_by="tag")

exc_subclass_count_fraction = exc_subclass_ct.normalize(by="pre", values="n_syn")
exc_subclass_size_fraction = exc_subclass_ct.normalize(by="pre", values="size_sum")
exc_subclass_count_fraction.df


## Target population counts

This is the denominator Polars step: count cells in the supplied population, independently of which cells receive synapses. The population keys match the aggregation keys above. The reserved __unlabeled__ bucket and ` / ` separator should not be used as literal taxonomy labels.


In [ ]:
population_cells = cells.filter(pl.col("class").is_in(["exc", "inh"])).with_columns(
    pl.col("subclass").cast(pl.String).fill_null("__unlabeled__").alias("target_subclass"),
    pl.concat_str(
        pl.col("class").cast(pl.String),
        pl.col("subclass").cast(pl.String).fill_null("__unlabeled__"),
        pl.col("cluster").cast(pl.String).fill_null("__unlabeled__"),
        separator=" / ",
    ).alias("target_cluster"),
)
class_population = population_cells.group_by("class").agg(pl.len().alias("n_targets"))
exc_subclass_population = population_cells.filter(pl.col("class") == "exc").group_by("target_subclass").agg(pl.len().alias("n_targets"))
inh_subclass_population = population_cells.filter(pl.col("class") == "inh").group_by("target_subclass").agg(pl.len().alias("n_targets"))
exc_cluster_population = population_cells.filter(pl.col("class") == "exc").group_by("target_cluster").agg(pl.len().alias("n_targets"))
inh_cluster_population = population_cells.filter(pl.col("class") == "inh").group_by("target_cluster").agg(pl.len().alias("n_targets"))
exc_subclass_population


## Population-adjusted allocation with Trajan

Attach population counts only on the type-bearing post axis, then define count/cell and total-size/cell. Normalize those rates over each presynaptic neuron's visible types. Types with no observations contribute zero to these totals; they are inserted explicitly in the final wide export.

These are abundance-adjusted descriptive allocations. Dendritic exposure, reconstruction completeness, and spatial opportunity require additional denominators. A neuron with zero total measured size has undefined size fractions (Trajan currently returns NaN); the export converts these to null.


In [ ]:
# First do one table explicitly; inspect each intermediate object.
exc_subclass_ct.add_annotation(
    "target_population", exc_subclass_population,
    cell_id_col="target_subclass", side="post",
)
exc_subclass_ct.add_expression("count_per_target", pl.col("n_syn") / pl.col("n_targets_post"))
exc_subclass_ct.add_expression("size_per_target", pl.col("size_sum") / pl.col("n_targets_post"))

exc_subclass_adjusted_count = exc_subclass_ct.normalize(by="pre", values="count_per_target")
exc_subclass_adjusted_size = exc_subclass_ct.normalize(by="pre", values="size_per_target")
exc_subclass_adjusted_count.df


In [ ]:
tables = {
    "class": (class_ct, class_population),
    "exc_subclass": (exc_subclass_ct, exc_subclass_population),
    "inh_subclass": (inh_subclass_ct, inh_subclass_population),
    "exc_cluster": (exc_cluster_ct, exc_cluster_population),
    "inh_cluster": (inh_cluster_ct, inh_cluster_population),
}

# Apply the identical annotation/expression pattern to the remaining tables.
# The existence check also makes rerunning this cell harmless.
for ct, population in tables.values():
    if "n_targets_post" not in ct.build_lazy().collect_schema().names():
        ct.add_annotation("target_population", population, cell_id_col=population.columns[0], side="post")
        ct.add_expression("count_per_target", pl.col("n_syn") / pl.col("n_targets_post"))
        ct.add_expression("size_per_target", pl.col("size_sum") / pl.col("n_targets_post"))


## Export one row per neuron

Trajan performs aggregation, annotation, normalization, and dense export. This final Polars step prefixes and joins the feature blocks. Counts/rates for unconnected types are zero. Means and fractions for neurons with no usable denominator are null. Tag fractions are exported per target group, including their explicit null bucket.

The output includes neurons present in `analysis`, not cells with no presynaptic synapses at all. Restrict `tables` or `measures` if you only want a few features.


In [ ]:
summary = baseline
for name, (ct, population) in tables.items():
    measures = {
        "n_syn": (ct, "n_syn", True),
        "mean_size": (ct, "mean_size", False),
        "count_fraction": (ct.normalize(by="pre", values="n_syn"), "fraction", False),
        "size_fraction": (ct.normalize(by="pre", values="size_sum"), "fraction", False),
        "count_per_target": (ct, "count_per_target", True),
        "size_per_target": (ct, "size_per_target", True),
        "adjusted_count_fraction": (ct.normalize(by="pre", values="count_per_target"), "fraction", False),
        "adjusted_size_fraction": (ct.normalize(by="pre", values="size_per_target"), "fraction", False),
    }
    target_labels = population[population.columns[0]].cast(pl.String).to_list()
    for metric, (table, value, zero_without_output) in measures.items():
        fill = float("nan") if metric == "mean_size" else 0
        matrix = table.to_dense(values=value, fill_value=fill)
        # Add types absent from the entire observed table.
        for label in target_labels:
            if label not in matrix.columns:
                matrix = matrix.with_columns(pl.lit(fill).alias(label))
        renamed = {label: f"{name}__{label}__{metric}" for label in target_labels}
        matrix = matrix.select(pre, *target_labels).rename(renamed)
        summary = summary.join(matrix, on=pre, how="left")
        if renamed:
            summary = summary.with_columns(pl.col(list(renamed.values())).fill_nan(None))
            if zero_without_output:
                summary = summary.with_columns(pl.col(list(renamed.values())).fill_null(0))

    # Tag struct -> one block of count fractions for each population-defined type.
    for label in target_labels:
        tag_rows = ct.filter(pl.col(ct.post_col).cast(pl.String) == label).df.select(pre, "tag").unnest("tag")
        tag_columns = [c for c in tag_rows.columns if c != pre]
        if tag_columns:
            tag_rows = tag_rows.rename({c: f"{name}__{label}__tag_{c}_fraction" for c in tag_columns})
            summary = summary.join(tag_rows, on=pre, how="left")

summary.sort(pre)
